# 03 — Separar y filtrar autores UNAM

Este notebook transforma la unión canónica en una estructura **publicación + autor UNAM**.

Principios de esta fase:

- conserva las 15 columnas canónicas y los índices originales;
- no crea `Base_origen`, no deduplica, no completa metadatos y no reclasifica `Area`;
- conserva únicamente afiliaciones UNAM vinculadas al autor de esa publicación;
- una afiliación UNAM se coloca en `Afiliacion1`; dos distintas, una en cada columna;
- más de diez autores, afiliaciones o referencias exige revisión manual;
- no identifica a UNAM por sus siglas aisladas ni a una persona solo por su apellido;
- el control y otras representaciones son propuestas para revisar, no aprobaciones;
- se utilizan solamente **tres CSV** en `04_Limpieza/01_Internos_unam`:
  `clasificacion_afiliaciones_unam.csv`, `casos_revision_manual.csv` y
  `autores_unam_separados_automatico.csv`;
- `autores_unam_separados_automatico.csv` contiene únicamente relaciones resueltas
  automáticamente: exactamente las 15 columnas originales, en el mismo orden,
  sin columnas auxiliares ni registros enviados a revisión manual;
- las decisiones y la base integrada quedan **en memoria**: no se crean archivos
  resueltos, finales, auditorías ni subcarpetas adicionales.

La salida automática se guarda aunque existan revisiones pendientes; no incorpora
los casos aprobados manualmente. Si no hay relaciones automáticas, conserva el
encabezado de las 15 columnas sin inventar registros.

El archivo manual adjunto contiene los 317 registros históricos disponibles, todos pendientes.
Al ejecutar sobre `Canonico_Union_Trabajo.csv`, el notebook obtiene la lista de revisión
correspondiente a **toda esa entrada**, conserva decisiones compatibles y detecta versiones ajenas.
Una fila manual aprobada representa un autor: para conservar varios autores del mismo registro,
copia esa fila y completa un autor por copia sin cambiar sus 15 campos originales ni `ID_revision`.
No cambies el nombre canónico aquí: esa decisión pertenece al notebook 04.

## Importaciones

In [ ]:
import os
import re
import html
import csv
import io
import json
import hashlib
import unicodedata
from collections import defaultdict, Counter

import pandas as pd

## 1. RUTAS Y COLUMNAS

In [ ]:
archivo = "../02_modelo_canonico/03_union/Canonico_Union_Trabajo.csv"

archivo_tutora_xlsx = "../00_control/UNAM_Completo_Corregido.xlsx"
archivo_tutora_csv = "../00_control/UNAM_Completo_Corregido.csv"

carpeta_salida = "../04_Limpieza/01_Internos_unam"
archivo_clasificacion = f"{carpeta_salida}/clasificacion_afiliaciones_unam.csv"
revision_manual = f"{carpeta_salida}/casos_revision_manual.csv"
salida_automatica = f"{carpeta_salida}/autores_unam_separados_automatico.csv"

# Funciona al ejecutar desde notebooks/ o desde la raíz del repositorio en VS Code.
if not os.path.exists(archivo) and os.path.isdir("02_modelo_canonico"):
    archivo = archivo[3:]
    archivo_tutora_xlsx = archivo_tutora_xlsx[3:]
    archivo_tutora_csv = archivo_tutora_csv[3:]
    carpeta_salida = carpeta_salida[3:]
    archivo_clasificacion = archivo_clasificacion[3:]
    revision_manual = revision_manual[3:]
    salida_automatica = salida_automatica[3:]

# False: nunca reemplaza un CSV distinto. Los resultados nuevos quedan en memoria.
# True: autoriza actualizar SOLO los tres CSV anteriores, preservando decisiones.
# Guarda antes tu versión en GitHub Desktop. Nunca se reescribe la unión ni el control.
actualizar_archivos = True

columnas = [
    "Fuente_origen", "indice", "Titulo", "Año", "Autor_norm",
    "Afiliacion1", "Afiliacion2", "ISBN", "ISSN", "Doi",
    "URL", "Area", "SubArea", "Keywords", "Abstract"
]
columnas_clasificacion = ["Afiliacion_original", "Estado_UNAM", "Evidencia"]
columnas_revision = columnas + [
    "ID_revision", "Motivo_revision", "Propuestas_no_aprobadas", "Decision",
    "Posicion_autor", "Autor_UNAM", "Afiliacion1_UNAM", "Afiliacion2_UNAM",
    "Revision_completa", "Evidencia", "URL_evidencia", "Comentario"
]
columnas_decision = [
    "Decision", "Posicion_autor", "Autor_UNAM", "Afiliacion1_UNAM",
    "Afiliacion2_UNAM", "Revision_completa", "Evidencia", "URL_evidencia", "Comentario"
]

## 2. FUNCIONES GENERALES

In [ ]:
def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()


def decodificar_html(texto, max_iter=10):
    """Decodifica HTML antes de separar por ';'."""
    texto = "" if texto is None else str(texto)

    for _ in range(max_iter):
        nuevo = html.unescape(texto)
        if nuevo == texto:
            break
        texto = nuevo

    return unicodedata.normalize("NFC", texto).strip()


def separar_punto_coma(texto):
    texto = decodificar_html(texto)
    return [x.strip() for x in texto.split(";") if x.strip()]


def sin_acentos(texto):
    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )


def normalizar_texto(texto):
    """Solo para comparar; no reemplaza el valor guardado."""
    texto = sin_acentos(decodificar_html(texto)).lower()
    texto = re.sub(r"[^a-z0-9]+", " ", texto)
    return " ".join(texto.split())


def normalizar_doi(doi):
    doi = decodificar_html(doi).lower().strip()
    doi = re.sub(r"^https?://(?:dx\.)?doi\.org/", "", doi)
    doi = re.sub(r"^doi:\s*", "", doi)
    return re.sub(r"\s+", "", doi)


def normalizar_titulo(titulo):
    return normalizar_texto(titulo)


def unicos_no_vacios(valores):
    resultado = []
    vistos = set()

    for valor in valores:
        valor = decodificar_html(valor)
        clave = decodificar_html(valor).casefold()

        if valor and clave and clave not in vistos:
            resultado.append(valor)
            vistos.add(clave)

    return resultado


def leer_csv(ruta, esquema=None):
    with open(ruta, "rb") as f:
        if f.read(100).startswith(b"version https://git-lfs.github.com/spec/"):
            raise ValueError(f"{ruta}: es un puntero Git LFS, no el CSV de datos.")
    tabla = pd.read_csv(ruta, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    adicionales = [c for c in tabla if c.startswith("Unnamed:")]
    for c in adicionales:
        if tabla[c].str.strip().ne("").any():
            raise ValueError(f"{ruta}: {c} contiene información; no se elimina.")
    if adicionales:
        print(f"AVISO: {ruta}: {len(adicionales)} columnas vacías ignoradas en memoria.")
        tabla = tabla.drop(columns=adicionales)
    if esquema is not None and list(tabla.columns) != esquema:
        raise ValueError(f"{ruta}: esquema inesperado: {list(tabla.columns)}")
    return tabla


def preparar_csv(tabla):
    if not all(isinstance(x, str) for x in tabla.to_numpy().ravel()):
        raise ValueError("Todos los valores de los CSV deben ser texto.")
    texto = tabla.to_csv(index=False, lineterminator="\n", quoting=csv.QUOTE_ALL)
    relectura = pd.read_csv(io.StringIO(texto), dtype=str, keep_default_na=False)
    if not relectura.equals(tabla.reset_index(drop=True)):
        raise ValueError("La serialización CSV cambia los datos o su esquema.")
    return texto.encode("utf-8-sig")


def guardar_csv(tabla, ruta):
    contenido = preparar_csv(tabla)
    if os.path.exists(ruta):
        existente = pd.read_csv(
            ruta, dtype=str, keep_default_na=False, encoding="utf-8-sig"
        )
        if existente.equals(tabla.reset_index(drop=True)):
            print("Sin cambios; esquema y datos verificados:", ruta)
            return True
        if not actualizar_archivos:
            print("NO se sobrescribió:", ruta)
            print("Resultado nuevo en memoria. Para guardarlo, activa actualizar_archivos.")
            return False
    # No se generan temporales, copias comprimidas ni otros archivos.
    modo = "wb" if os.path.exists(ruta) else "xb"
    with open(ruta, modo) as f:
        f.write(contenido)
    relectura = pd.read_csv(
        ruta, dtype=str, keep_default_na=False, encoding="utf-8-sig"
    )
    if list(relectura.columns) != list(tabla.columns) or relectura.shape != tabla.shape:
        raise ValueError(f"{ruta}: la relectura cambió las filas, columnas o su orden.")
    if not relectura.equals(tabla.reset_index(drop=True)):
        raise ValueError(f"{ruta}: la relectura no coincide con los datos originales.")
    print("Guardado y releído:", ruta, tabla.shape)
    return True


def firma_registro(fila):
    datos = [fila[c] for c in columnas]
    return hashlib.sha256(json.dumps(datos, ensure_ascii=False).encode("utf-8")).hexdigest()

## 3. REFERENCIAS NUMÉRICAS Y COMPARACIÓN DE NOMBRES

In [ ]:
# Scopus Author ID: (57191896522)
SCOPUS_ID_RE = re.compile(r"\s*\((\d{7,12})\)\s*$")

# Referencias institucionales: (1), (1,2), (1, 2, 3)
REF_AUTOR_RE = re.compile(r"\s*\((\d{1,3}(?:\s*,\s*\d{1,3})*)\)\s*$")
MARCADOR_AFILIACION_RE = re.compile(r"(?<!\w)\((\d{1,3})\)\s*")


def quitar_scopus_id(autor):
    return SCOPUS_ID_RE.sub("", decodificar_html(autor)).strip()


def referencias_autor(autor):
    m = REF_AUTOR_RE.search(decodificar_html(autor))
    return [x.strip() for x in m.group(1).split(",")] if m else []


def quitar_referencias_autor(autor):
    return REF_AUTOR_RE.sub("", decodificar_html(autor)).strip()


def nombre_separado(autor, fuente):
    # Retirar un ID largo solo cuando el registro es Scopus.
    autor = quitar_scopus_id(autor) if fuente == "Scopus" else decodificar_html(autor)
    return quitar_referencias_autor(autor)


def mapa_afiliaciones_numeradas(afiliacion1, afiliacion2=""):
    """Devuelve catálogo y problemas: nunca sobrescribe números contradictorios."""
    resultado, problemas = {}, []
    textos = [decodificar_html(afiliacion1), decodificar_html(afiliacion2)]
    hay_catalogo = any(MARCADOR_AFILIACION_RE.search(t) for t in textos)
    for texto in textos:
        if not texto:
            continue
        encontrados = list(MARCADOR_AFILIACION_RE.finditer(texto))
        if not encontrados:
            if hay_catalogo:
                problemas.append("columna sin referencias junto a catálogo numerado")
            continue
        if texto[:encontrados[0].start()].strip(" ;"):
            problemas.append("texto sin referencia antes del catálogo")
        for i, m in enumerate(encontrados):
            fin = encontrados[i + 1].start() if i + 1 < len(encontrados) else len(texto)
            afiliacion = texto[m.end():fin].strip(" ;")
            ref = m.group(1)
            if not afiliacion:
                problemas.append("afiliación numerada vacía")
            if ref in resultado and resultado[ref] != afiliacion:
                problemas.append("referencia institucional con valores distintos")
            else:
                resultado[ref] = afiliacion
    return resultado, problemas


def tokens_nombre(nombre):
    # Comparación temporal; no sustituye la grafía del nombre guardado.
    nombre = quitar_referencias_autor(quitar_scopus_id(nombre))
    if "," in nombre:
        apellido, nombres = [x.strip() for x in nombre.split(",", 1)]
        nombre = nombres + " " + apellido
    return re.findall(r"[^\W_]+", sin_acentos(nombre).casefold())


def nombres_compatibles(a, b):
    """Nombre completo o iniciales compatibles; nunca solo el apellido.

    La unicidad se comprueba además contra TODOS los autores del registro.
    No admite subsecuencias, fuzzy matching ni menos de dos componentes.
    """
    ta, tb = tokens_nombre(a), tokens_nombre(b)
    if len(ta) != len(tb) or len(ta) < 2:
        return False
    if not any(x == y and len(x) > 1 for x, y in zip(ta, tb)):
        return False
    return all(
        x == y or (len(x) == 1 and y.startswith(x)) or (len(y) == 1 and x.startswith(y))
        for x, y in zip(ta, tb)
    )

## 4. SCOPUS, WOS Y UNAM

In [ ]:
# Son criterios de reconocimiento textual, no una verificación de publicaciones.
UNAM_COMPLETA = re.compile(
    r"\b(?:universidad nacional autonoma de mexico|"
    r"national autonomous university of mexico|universite nationale autonome du mexique)\b"
)
EXTERNAS_EXPLICITAS = re.compile(
    r"\b(?:bilkent|universidad nacional de misiones|universidad nacional de moquegua|"
    r"university of namibia|universidad autonoma de la ciudad de mexico|"
    r"universidad autonoma metropolitana|instituto politecnico nacional|"
    r"national polytechnic institute|cinvestav|tecnologico de monterrey|itesm|"
    r"instituto nacional de medicina genomica|inmegen|"
    r"instituto nacional de enfermedades respiratorias|"
    r"centre for research in mathematics|centro de investigacion en matematicas)\b"
)
COLISION_UNAM = re.compile(
    r"\bunam\b.*\b(?:misiones|obera|argentina|moquegua|namibia|bilkent)\b|"
    r"\b(?:misiones|moquegua|bilkent|namibia)\b.*\bunam\b"
)
UNIVERSIDAD = re.compile(
    r"\b(?:universidad|university|universite|universitat|universita|universidade|universiti|universiteit)\b"
)


def clasificar_literal(afiliacion):
    # Un fragmento no reconocido después de ';' podría ser otra institución.
    # partir_afiliacion separa los casos inequívocos antes de clasificar la relación.
    if len(separar_punto_coma(afiliacion)) > 1:
        return "MIXTA", "varios fragmentos separados por punto y coma; requiere delimitación"
    texto = normalizar_texto(afiliacion)
    completa = bool(UNAM_COMPLETA.search(texto))
    externa = bool(EXTERNAS_EXPLICITAS.search(texto) or COLISION_UNAM.search(texto))
    if completa:
        resto = UNAM_COMPLETA.sub("", texto)
        resto = re.sub(r"\b(?:avenida|av|ave|avenue) universidad\b|\buniversity city\b", "", resto)
        otra = re.search(r"\bhospital general|\btecnologico nacional de mexico|\binstituto tecnologico de morelia|\bcnrs\b|\binserm\b", texto)
        if externa or UNIVERSIDAD.search(resto) or otra or len(UNAM_COMPLETA.findall(texto)) > 1:
            return "MIXTA", "varias instituciones en un mismo fragmento; requiere separarlas"
        return "UNAM", "denominación completa de la Universidad Nacional Autónoma de México"
    siglas = bool(re.search(r"\bunam\b|\bu n a m\b", texto))
    if siglas and UNIVERSIDAD.search(texto) and not COLISION_UNAM.search(texto):
        return "MIXTA", "siglas UNAM junto a otra denominación universitaria"
    if externa:
        return "EXTERNO", "denominación explícita de otra institución o siglas homónimas"
    return "AMBIGUA", "siglas aisladas o denominación sin evidencia institucional suficiente"


def clasificar_afiliacion(afiliacion):
    clave = decodificar_html(afiliacion).casefold()
    estado, evidencia = clasificar_literal(afiliacion)
    decisiones = clasificacion_por_texto.get(clave, [])
    if len({r["Estado_UNAM"] for r in decisiones}) > 1:
        return "AMBIGUA", "clasificaciones contradictorias de la misma cadena"
    if not decisiones:
        return estado, evidencia
    decision = decisiones[0]
    propuesto = decision["Estado_UNAM"]
    if propuesto in {"AMBIGUA", "MIXTA"}:
        return propuesto, decision["Evidencia"]
    if estado in {"UNAM", "EXTERNO"}:
        if estado != propuesto:
            return "AMBIGUA", "conflicto entre denominación y clasificación"
        return estado, decision["Evidencia"]
    if estado == "MIXTA":
        return estado, evidencia
    # Excepción institucional revisada: documentar con VERIFICADO: y una URL.
    prueba = decision["Evidencia"].strip()
    generica = set(normalizar_texto(afiliacion).split()) <= {
        "unam", "u", "n", "a", "m", "mexico", "city", "ciudad", "de", "cdmx", "df"
    }
    if not generica and prueba.startswith("VERIFICADO:") and re.search(r"https?://\S+", prueba):
        return propuesto, prueba
    return "AMBIGUA", "clasificación histórica sin validación suficiente; revisar"


def es_afiliacion_unam(afiliacion):
    return clasificar_afiliacion(afiliacion)[0] == "UNAM"


def partir_afiliacion(afiliacion):
    partes = separar_punto_coma(afiliacion)
    if len(partes) > 1 and all(clasificar_afiliacion(p)[0] in {"UNAM", "EXTERNO"} for p in partes):
        return partes
    return [decodificar_html(afiliacion)] if afiliacion.strip() else []


def afiliaciones_scopus(entrada_autor, catalogo):
    """Localiza la etiqueta nominal y las instituciones literales del catálogo."""
    entrada = decodificar_html(entrada_autor)
    hallazgos = []
    for institucion in catalogo:
        for m in re.finditer(re.escape(institucion), entrada):
            i, j = m.span()
            if (i == 0 or entrada[i-1] in " ,;") and (j == len(entrada) or entrada[j] in " ,;."):
                hallazgos.append((i, j, institucion))
    hallazgos = [x for x in hallazgos if not any(
        y[0] <= x[0] and y[1] >= x[1] and (y[0], y[1]) != (x[0], x[1])
        for y in hallazgos
    )]
    hallazgos = sorted(set(hallazgos))
    if not hallazgos:
        return "", [], False
    etiqueta = entrada[:hallazgos[0][0]].strip(" ,;")
    resto = list(entrada[hallazgos[0][0]:])
    for i, j, _ in hallazgos:
        for k in range(i-hallazgos[0][0], j-hallazgos[0][0]):
            resto[k] = " "
    completo = not "".join(resto).strip(" ,;.\t\n")
    return etiqueta, [x[2] for x in hallazgos], completo


def corresponding_wos(texto):
    resultado = []
    for segmento in separar_punto_coma(texto):
        m = re.fullmatch(r"(.*?)\s*\(corresponding author\)\s*,\s*(.+)", segmento, re.I | re.S)
        if m:
            resultado.append((m.group(1).strip(), m.group(2).strip()))
    return resultado

## 5. CARGAR DATOS

In [ ]:
if not os.path.exists(archivo):
    raise FileNotFoundError(f"Falta la entrada original: {archivo}. No se usa una base posterior.")
hash_antes = sha256(archivo)
df = leer_csv(archivo, columnas)

# Detectar daños; esta fase NO inventa dígitos ni cambia los identificadores.
for campo in ["indice", "ISBN", "ISSN"]:
    malos = df[campo].str.contains(r"(?<![A-Za-z0-9])\d+(?:[.,]\d+)?[eE][+-]?\d+(?![A-Za-z0-9])", regex=True)
    if malos.any():
        raise ValueError(f"{campo}: notación científica en filas {(df.index[malos] + 1).tolist()[:20]}. Revisar la entrada original.")

df["_row_id"] = range(len(df))
firmas = df.apply(firma_registro, axis=1).astype(str) if len(df) else pd.Series(index=df.index, dtype=str)
ocurrencias = firmas.groupby(firmas, sort=False).cumcount() + 1
df["_id_revision"] = firmas + "_" + ocurrencias.astype(str)

if not os.path.exists(archivo_clasificacion):
    raise FileNotFoundError(f"Coloca clasificacion_afiliaciones_unam.csv en {carpeta_salida}.")
clasificacion = leer_csv(archivo_clasificacion, columnas_clasificacion)
if clasificacion["Afiliacion_original"].str.strip().eq("").any():
    raise ValueError("La clasificación contiene afiliaciones vacías.")
if clasificacion["Afiliacion_original"].duplicated().any():
    raise ValueError("La clasificación tiene claves exactas duplicadas.")
if not set(clasificacion["Estado_UNAM"]) <= {"UNAM", "EXTERNO", "AMBIGUA", "MIXTA"}:
    raise ValueError("Estado institucional no permitido.")
if clasificacion["Evidencia"].str.strip().eq("").any():
    raise ValueError("Cada clasificación requiere evidencia o motivo de revisión.")
clasificacion_por_texto = defaultdict(list)
for registro in clasificacion.to_dict("records"):
    clasificacion_por_texto[decodificar_html(registro["Afiliacion_original"]).casefold()].append(registro)

# La tutora es evidencia auxiliar: no es obligatoria ni acredita afiliaciones por sí sola.
columnas_tutora = ["Titulo", "Año", "Autor_norm", "Afiliacion1", "Afiliacion2", "Doi"]
tutora = pd.DataFrame(columns=columnas_tutora)
if os.path.exists(archivo_tutora_csv):
    tutora = leer_csv(archivo_tutora_csv)
elif os.path.exists(archivo_tutora_xlsx):
    tutora = pd.read_excel(archivo_tutora_xlsx, dtype=str, keep_default_na=False)
if not set(columnas_tutora) <= set(tutora.columns):
    raise ValueError("El archivo de control no contiene las columnas necesarias.")

revision_anterior = pd.DataFrame(columns=columnas_revision)
if os.path.exists(revision_manual):
    revision_anterior = leer_csv(revision_manual)
    if list(revision_anterior.columns) == columnas:
        print("Revisión histórica de 15 columnas: se incorpora como PENDIENTE, sin aprobar decisiones.")
        for c in columnas_revision[len(columnas):]:
            revision_anterior[c] = "PENDIENTE" if c == "Decision" else ""
    elif list(revision_anterior.columns) != columnas_revision:
        raise ValueError("casos_revision_manual.csv tiene un esquema incompatible; no se sobrescribe.")
    for i, fila in revision_anterior.iterrows():
        firma = firma_registro(fila)
        posibles = df.loc[firmas.eq(firma), "_id_revision"].tolist()
        if fila["ID_revision"]:
            if fila["ID_revision"] not in posibles:
                raise ValueError(f"Revisión de otra entrada o metadatos originales alterados: fila {i+1}.")
        elif len(posibles) == 1:
            revision_anterior.at[i, "ID_revision"] = posibles[0]
        else:
            raise ValueError(f"No se puede vincular inequívocamente la revisión histórica, fila {i+1}.")

print("Filas de entrada:", len(df))
print("Columnas canónicas:", len(columnas))
print("Filas del control:", len(tutora))
print("SHA-256 de la entrada (no es un archivo adicional):", hash_antes)

## 6. CASOS GRANDES: >10 AUTORES / AFILIACIONES / REFERENCIAS

In [ ]:
def contar_estructura(fila):
    autores = separar_punto_coma(fila["Autor_norm"])
    catalogo, _ = mapa_afiliaciones_numeradas(fila["Afiliacion1"], fila["Afiliacion2"])
    if catalogo:
        n_afiliaciones = max(len(catalogo), sum(max(1, len(separar_punto_coma(a))) for a in catalogo.values()))
    elif fila["Fuente_origen"].strip() == "Scopus":
        # El catálogo y la lista de correspondencias no son afiliaciones adicionales.
        n_afiliaciones = len(separar_punto_coma(fila["Afiliacion1"]))
    else:
        n_afiliaciones = len(unicos_no_vacios([
            a for c in ["Afiliacion1", "Afiliacion2"] for a in separar_punto_coma(fila[c])
        ]))
    # Contar todas las referencias, no solo sus números distintos.
    n_referencias = sum(len(referencias_autor(a)) for a in autores)
    return pd.Series({
        "_n_autores": len(autores),
        "_n_afiliaciones": n_afiliaciones,
        "_n_referencias": n_referencias,
    })

conteos = df.apply(contar_estructura, axis=1) if len(df) else pd.DataFrame(
    columns=["_n_autores", "_n_afiliaciones", "_n_referencias"]
)
df = pd.concat([df, conteos], axis=1)
df["_caso_grande"] = df[["_n_autores", "_n_afiliaciones", "_n_referencias"]].gt(10).any(axis=1)
print("Casos grandes enviados directamente a revisión:", int(df["_caso_grande"].sum()))

## 7. PRIMERA PASADA: SEPARACIÓN AUTOR-AFILIACIÓN

In [ ]:
problemas_por_fila = defaultdict(list)


def separar_fila(fila):
    row_id = fila["_row_id"]
    originales = separar_punto_coma(fila["Autor_norm"])
    fuente = fila["Fuente_origen"].strip()
    autores = [nombre_separado(a, fuente) for a in originales]
    base = {
        "row_id": row_id, "Fuente_origen": fila["Fuente_origen"],
        "indice": fila["indice"], "Titulo": fila["Titulo"], "Año": fila["Año"], "Doi": fila["Doi"]
    }
    relaciones_fila = [dict(base, Autor=a, Posicion_autor=str(i+1), Afiliaciones=[],
        Estado_separacion="PENDIENTE", Evidencia="sin relación explícita") for i, a in enumerate(autores)]
    if not autores:
        problemas_por_fila[row_id].append("lista de autores vacía")
        return relaciones_fila
    if len({normalizar_texto(a) for a in autores}) != len(autores):
        problemas_por_fila[row_id].append("nombres repetidos; requiere desambiguación")
    for a in autores:
        if len(tokens_nombre(a)) < 2 or re.search(r"[()@]|\bet\s+al\b|\s(?:and|&|y)\s", a, re.I):
            problemas_por_fila[row_id].append("nombre incompleto o estructura de autores no resuelta")
    # Los grandes no se resuelven automáticamente, aunque tengan referencias numéricas.
    if fila["_caso_grande"]:
        problemas_por_fila[row_id].append(">10 autores/afiliaciones/referencias")
        return relaciones_fila

    catalogo, errores = mapa_afiliaciones_numeradas(fila["Afiliacion1"], fila["Afiliacion2"])
    problemas_por_fila[row_id].extend(errores)
    hay_referencias = any(referencias_autor(a) for a in originales)

    # A. Correspondencias numéricas explícitas: prioridad sobre el nombre de la fuente.
    if catalogo:
        for original, relacion in zip(originales, relaciones_fila):
            refs = referencias_autor(original)
            validas = bool(refs) and all(r in catalogo and catalogo[r] for r in refs)
            if len(refs) != len(set(refs)):
                validas = False
            if validas and not errores:
                relacion["Afiliaciones"] = [p for r in refs for p in partir_afiliacion(catalogo[r])]
                relacion["Estado_separacion"] = "COMPLETO"
                relacion["Evidencia"] = "referencias explícitas del autor: " + original
        return relaciones_fila
    if hay_referencias:
        problemas_por_fila[row_id].append("referencias de autor sin catálogo institucional")
        return relaciones_fila

    # B. Scopus: etiqueta nominal única + catálogo literal; no listas por posición.
    if fuente == "Scopus":
        instituciones = separar_punto_coma(fila["Afiliacion1"])
        usadas = defaultdict(int)
        for entrada in separar_punto_coma(fila["Afiliacion2"]):
            etiqueta, afiliaciones, completo = afiliaciones_scopus(entrada, instituciones)
            candidatos = [i for i, a in enumerate(autores) if nombres_compatibles(a, etiqueta)]
            if not completo or len(candidatos) != 1:
                problemas_por_fila[row_id].append("Scopus: correspondencia nominal ambigua o incompleta")
                continue
            i = candidatos[0]
            usadas[i] += 1
            relaciones_fila[i]["Afiliaciones"].extend(p for a in afiliaciones for p in partir_afiliacion(a))
            relaciones_fila[i]["Estado_separacion"] = "COMPLETO"
            relaciones_fila[i]["Evidencia"] = "Scopus etiqueta única y catálogo: " + entrada
        if any(n > 1 for n in usadas.values()):
            problemas_por_fila[row_id].append("Scopus: varias entradas para una etiqueta; verificar homónimos")
        return relaciones_fila

    # C. WoS: solo relaciones con nombre del corresponding author inequívoco.
    if fuente == "WoS":
        entradas = separar_punto_coma(fila["Afiliacion2"])
        pares = corresponding_wos(fila["Afiliacion2"])
        if len(pares) != len(entradas):
            problemas_por_fila[row_id].append("WoS: fragmentos sin enlace explícito")
        for etiqueta, afiliacion in pares:
            candidatos = [i for i, a in enumerate(autores) if nombres_compatibles(a, etiqueta)]
            if len(candidatos) != 1:
                problemas_por_fila[row_id].append("WoS: etiqueta nominal ambigua")
                continue
            relacion = relaciones_fila[candidatos[0]]
            relacion["Afiliaciones"].extend(partir_afiliacion(afiliacion))
            relacion["Estado_separacion"] = "COMPLETO"
            relacion["Evidencia"] = "WoS corresponding author: " + etiqueta + " | " + afiliacion
        return relaciones_fila

    # D. Sin enlace explícito: no asumir que las listas IEEE están alineadas.
    if fuente == "IEEE":
        problemas_por_fila[row_id].append("IEEE: listas paralelas sin prueba nominal o numérica")
    elif len(autores) == 1:
        problemas_por_fila[row_id].append("único autor registrado no demuestra que todas las afiliaciones sean suyas")
    else:
        problemas_por_fila[row_id].append("multiautor sin relación explícita")
    return relaciones_fila


registros = []
for _, fila in df.iterrows():
    registros.extend(separar_fila(fila))
columnas_relaciones = ["row_id", "Fuente_origen", "indice", "Titulo", "Año", "Doi",
    "Autor", "Posicion_autor", "Afiliaciones", "Estado_separacion", "Evidencia"]
relaciones = pd.DataFrame(registros, columns=columnas_relaciones)
relaciones["Afiliaciones"] = relaciones["Afiliaciones"].map(unicos_no_vacios)
print("Apariciones de autor extraídas:", len(relaciones))
print(relaciones["Estado_separacion"].value_counts())

## 8. SEGUNDA PASADA: MISMA PUBLICACIÓN EN OTRA FUENTE

In [ ]:
# Solo recopilar propuestas: no copiar afiliaciones, no completar relaciones pendientes.
por_doi = defaultdict(list)
por_titulo_anio = defaultdict(list)
for i, r in relaciones[relaciones["Estado_separacion"].eq("COMPLETO")].iterrows():
    doi = normalizar_doi(r["Doi"])
    if doi:
        por_doi[doi].append(i)
    clave = (normalizar_titulo(r["Titulo"]), r["Año"].strip())
    if clave[0]:
        por_titulo_anio[clave].append(i)

propuestas_por_fila = defaultdict(list)
for _, r in relaciones[relaciones["Estado_separacion"].ne("COMPLETO")].iterrows():
    doi = normalizar_doi(r["Doi"])
    candidatos = por_doi.get(doi, []) if doi else por_titulo_anio.get(
        (normalizar_titulo(r["Titulo"]), r["Año"].strip()), [])
    for j in candidatos:
        otro = relaciones.loc[j]
        if otro["row_id"] == r["row_id"] or not nombres_compatibles(r["Autor"], otro["Autor"]):
            continue
        if doi and normalizar_titulo(r["Titulo"]) != normalizar_titulo(otro["Titulo"]):
            continue
        propuestas_por_fila[r["row_id"]].append(
            "NO APROBADO. Otra representación: " + otro["Autor"] + " | " +
            "; ".join(otro["Afiliaciones"]) + " | " + otro["Fuente_origen"] + " / " + otro["indice"]
        )
print("Registros con propuestas de otra representación:", len(propuestas_por_fila))

## 9. TERCERA PASADA: ARCHIVO DE LA TUTORA

In [ ]:
tutora = tutora.copy()
tutora["_doi_norm"] = tutora["Doi"].map(normalizar_doi)
tutora["_titulo_norm"] = tutora["Titulo"].map(normalizar_titulo)
tutora_por_doi = defaultdict(list)
tutora_por_titulo_anio = defaultdict(list)
for i, fila in tutora.iterrows():
    if fila["_doi_norm"]:
        tutora_por_doi[fila["_doi_norm"]].append(i)
    if fila["_titulo_norm"]:
        tutora_por_titulo_anio[(fila["_titulo_norm"], fila["Año"].strip())].append(i)

coincidencias_control = 0
for _, r in relaciones.iterrows():
    doi = normalizar_doi(r["Doi"])
    candidatos = tutora_por_doi.get(doi, []) if doi else tutora_por_titulo_anio.get(
        (normalizar_titulo(r["Titulo"]), r["Año"].strip()), [])
    for j in candidatos:
        t = tutora.loc[j]
        if normalizar_titulo(r["Titulo"]) != t["_titulo_norm"]:
            continue
        if not nombres_compatibles(r["Autor"], t["Autor_norm"]):
            continue
        coincidencias_control += 1
        propuestas_por_fila[r["row_id"]].append(
            "NO APROBADO. Control tutora: " + t["Autor_norm"] + " | " +
            t["Afiliacion1"] + " | " + t["Afiliacion2"]
        )
# No existe Coincide_tutora => UNAM. Ninguna propuesta cambia Estado_separacion.
print("Coincidencias del control, solo como evidencia auxiliar:", coincidencias_control)

## 10. FILAS QUE VAN A REVISIÓN MANUAL

In [ ]:
motivos_revision = defaultdict(list)
for row_id, problemas in problemas_por_fila.items():
    motivos_revision[row_id].extend(problemas)

# Conservar la revisión histórica identificada, sin aprobar sus antiguas soluciones.
ids_previos = set(revision_anterior["ID_revision"])
for _, fila in df.iterrows():
    if fila["_id_revision"] in ids_previos:
        motivos_revision[fila["_row_id"]].append("registro ya incluido en revisión manual")
    if fila["_caso_grande"]:
        motivos_revision[fila["_row_id"]].append(">10 autores/afiliaciones/referencias")

# Alertas de discrepancias detectadas en los adjuntos, no listas negras de personas.
# Las dos publicaciones tuvieron asignaciones IIMAS que requieren evidencia por autor.
alertas_titulo = {
    normalizar_titulo("Finding the Set of Nearly Optimal Solutions of a Multiobjective Optimization Problem"),
    normalizar_titulo("Screening and Structural Characterization of Heat Shock Response Elements (HSEs) in Entamoeba histolytica Promoters")
}
# Estos cuatro registros históricos no tuvieron un cierre manual documentado.
alertas_doi = {"10.1016/j.neo.2024.100987", "10.3389/fonc.2024.1355335",
    "10.1145/3700889", "10.1109/cce62852.2024.10771009"}
for _, fila in df.iterrows():
    if normalizar_titulo(fila["Titulo"]) in alertas_titulo or normalizar_doi(fila["Doi"]) in alertas_doi:
        motivos_revision[fila["_row_id"]].append("alerta histórica: verificar autor-publicación; no exclusión automática")

filas_pendientes, filas_mas_dos = set(), set()
clasificaciones_relacion = []
for _, r in relaciones.iterrows():
    estados = [clasificar_afiliacion(a)[0] for a in r["Afiliaciones"]]
    clasificaciones_relacion.append(estados)
    if r["Estado_separacion"] != "COMPLETO" or not estados:
        filas_pendientes.add(r["row_id"])
        motivos_revision[r["row_id"]].append("separación autor-afiliación no resuelta")
    if any(e in {"AMBIGUA", "MIXTA"} for e in estados):
        motivos_revision[r["row_id"]].append("afiliación ambigua o varias instituciones en un campo")
    if estados.count("UNAM") > 2:
        filas_mas_dos.add(r["row_id"])
        motivos_revision[r["row_id"]].append("autor con más de dos afiliaciones UNAM")
relaciones["Clasificaciones"] = clasificaciones_relacion
motivos_revision = {i: sorted(set(v)) for i, v in motivos_revision.items() if v}
filas_revision = set(motivos_revision)
diagnostico_revision = pd.DataFrame([
    {"_row_id": i, "Motivo_revision": " | ".join(motivos_revision[i])}
    for i in sorted(filas_revision)
], columns=["_row_id", "Motivo_revision"])
print("Filas originales para revisión manual:", len(filas_revision))

## 11. CLASIFICAR UNAM / EXTERNO

In [ ]:
def clasificar_unam(fila):
    if fila["row_id"] in filas_revision:
        return "REVISION"
    if fila["Estado_separacion"] != "COMPLETO" or not fila["Clasificaciones"]:
        return "REVISION"
    if "UNAM" in fila["Clasificaciones"]:
        return "UNAM"
    if all(e == "EXTERNO" for e in fila["Clasificaciones"]):
        return "EXTERNO"
    return "REVISION"

relaciones["Estado_UNAM"] = relaciones.apply(clasificar_unam, axis=1) if len(relaciones) else pd.Series(dtype=str)
relaciones_automaticas = relaciones[~relaciones["row_id"].isin(filas_revision)].copy()
# Auditoría disponible en memoria, sin un CSV adicional.
auditoria_relaciones = relaciones.copy(deep=True)
print("Clasificación publicación + autor:")
print(relaciones["Estado_UNAM"].value_counts())

## 12. CONSTRUIR SALIDA AUTOMÁTICA

In [ ]:
relaciones_unam = relaciones_automaticas[relaciones_automaticas["Estado_UNAM"].eq("UNAM")].copy()
salida_trabajo = []
for _, r in relaciones_unam.iterrows():
    original = df.loc[r["row_id"], columnas].to_dict()
    afiliaciones = unicos_no_vacios([
        a for a, e in zip(r["Afiliaciones"], r["Clasificaciones"]) if e == "UNAM"
    ])
    if not 1 <= len(afiliaciones) <= 2:
        raise ValueError("Se intentó conservar un autor sin una o dos afiliaciones UNAM.")
    original["Autor_norm"] = r["Autor"]
    original["Afiliacion1"] = afiliaciones[0]
    original["Afiliacion2"] = afiliaciones[1] if len(afiliaciones) == 2 else ""
    original["_row_id"] = r["row_id"]
    salida_trabajo.append(original)

autores_unam_automatico = pd.DataFrame(salida_trabajo, columns=columnas + ["_row_id"])
# _row_id solo se usa en memoria para validar la procedencia de cada relación.
# La selección explícita excluye cualquier columna auxiliar del archivo automático.
salida_automatica_df = autores_unam_automatico.loc[:, columnas].copy().reset_index(drop=True)
print("Filas UNAM automáticas:", len(salida_automatica_df))

## 13. ÚNICO ARCHIVO DE REVISIÓN MANUAL

In [ ]:
# Los 15 campos originales nunca se editan para resolver un caso.
# Decision: PENDIENTE / CONSERVAR / EXCLUIR.
# CONSERVAR: una copia de la fila por autor, Posicion_autor (1, 2, ...),
# Autor_UNAM, una o dos afiliaciones UNAM, Evidencia y URL_evidencia.
# Revision_completa=SI certifica que se revisó el registro entero y que
# los autores no incluidos se descartan de forma documentada (explicar en Comentario).
# EXCLUIR: una sola fila con motivo y evidencia; no clasifica a toda la persona como externa.
# PENDIENTE: no produce filas finales y no se interpreta como una exclusión aprobada.

casos = []
for i in sorted(filas_revision):
    fila = df.loc[i]
    anteriores = revision_anterior[revision_anterior["ID_revision"].eq(fila["_id_revision"])]
    if anteriores.empty:
        anteriores = pd.DataFrame([{c: "" for c in columnas_revision}])
        anteriores["Decision"] = "PENDIENTE"
    for _, anterior in anteriores.iterrows():
        registro = {c: fila[c] for c in columnas}
        registro["ID_revision"] = fila["_id_revision"]
        registro["Motivo_revision"] = " | ".join(motivos_revision[i])
        propuestas = list(dict.fromkeys(propuestas_por_fila.get(i, [])))
        registro["Propuestas_no_aprobadas"] = " || ".join(propuestas)
        for c in columnas_decision:
            registro[c] = anterior[c]
        casos.append(registro)
casos_revision = pd.DataFrame(casos, columns=columnas_revision).fillna("").astype(str)

# Completar el mismo catálogo, únicamente con textos institucionales identificados.
conocidas = set(clasificacion["Afiliacion_original"])
nuevas = []
for afiliacion in unicos_no_vacios([a for lista in relaciones["Afiliaciones"] for a in lista]):
    if afiliacion not in conocidas:
        estado, motivo = clasificar_literal(afiliacion)
        nuevas.append({"Afiliacion_original": afiliacion, "Estado_UNAM": estado,
            "Evidencia": "Diagnóstico textual: " + motivo + ". No demuestra vínculo con un autor."})
        conocidas.add(afiliacion)
clasificacion_final = pd.concat([clasificacion, pd.DataFrame(nuevas, columns=columnas_clasificacion)], ignore_index=True)
print("Registros originales distintos en revisión:", casos_revision["ID_revision"].nunique())

## 14. VALIDACIONES

In [ ]:
columnas_inmutables = [c for c in columnas if c not in {"Autor_norm", "Afiliacion1", "Afiliacion2"}]


def validar_salida(tabla):
    if list(tabla.columns) != columnas + ["_row_id"]:
        raise ValueError("La tabla no conserva su esquema de trabajo.")
    for _, fila in tabla.iterrows():
        original = df.loc[fila["_row_id"]]
        if any(fila[c] != original[c] for c in columnas_inmutables):
            raise ValueError("Cambió un metadato protegido en la separación.")
        if not fila["Autor_norm"].strip() or ";" in fila["Autor_norm"]:
            raise ValueError("La salida debe tener exactamente un autor por fila.")
        if not fila["Afiliacion1"].strip():
            raise ValueError("Una fila conservada requiere al menos una afiliación UNAM.")
        if fila["Afiliacion2"] and decodificar_html(fila["Afiliacion1"]).casefold() == decodificar_html(fila["Afiliacion2"]).casefold():
            raise ValueError("Afiliacion1 y Afiliacion2 son idénticas.")

validar_salida(autores_unam_automatico)
if autores_unam_automatico["_row_id"].isin(filas_revision).any():
    raise ValueError("Un registro de revisión fue incluido también como automático.")
if len(columnas) != 15 or list(salida_automatica_df.columns) != columnas:
    raise ValueError("La salida automática debe tener exactamente las 15 columnas originales, en su orden.")
if any(c.startswith("Unnamed") or c.startswith("_") for c in salida_automatica_df.columns):
    raise ValueError("La salida automática contiene columnas auxiliares.")
if not salida_automatica_df.equals(autores_unam_automatico[columnas].reset_index(drop=True)):
    raise ValueError("La salida automática no coincide con las relaciones automáticas validadas.")
for campo in ["Afiliacion1", "Afiliacion2"]:
    for afiliacion in salida_automatica_df[campo]:
        if afiliacion and not es_afiliacion_unam(afiliacion):
            raise ValueError("La salida automática contiene una afiliación no confirmada como UNAM.")
if hash_antes != sha256(archivo):
    raise ValueError("Se modificó el archivo original durante el proceso.")
if list(clasificacion_final.columns) != columnas_clasificacion:
    raise ValueError("La clasificación tiene columnas adicionales.")
# Verifica en memoria la relectura exacta antes de cualquier escritura.
preparar_csv(clasificacion_final)
preparar_csv(casos_revision)
preparar_csv(salida_automatica_df)
print("Validaciones previas a escritura: OK")

## 15. INCORPORACIÓN OPCIONAL DE LA REVISIÓN MANUAL

In [ ]:
# Toda la revisión se resuelve dentro de casos_revision_manual.csv.
# No se requiere casos_revision_resueltos.csv ni otros formularios.
manual_aprobado, cierres_revision = [], []
ids_pendientes = set()
por_id = df.set_index("_id_revision", drop=False)
for identificador, grupo in casos_revision.groupby("ID_revision", sort=False):
    estados = set(grupo["Decision"])
    if not estados <= {"PENDIENTE", "CONSERVAR", "EXCLUIR"}:
        raise ValueError(f"{identificador}: Decision inválida.")
    if not set(grupo["Revision_completa"]) <= {"", "NO", "SI"}:
        raise ValueError(f"{identificador}: Revision_completa debe ser SI, NO o vacío.")
    if "PENDIENTE" in estados or not grupo["Revision_completa"].eq("SI").all():
        ids_pendientes.add(identificador)
        continue
    if len(estados) != 1:
        raise ValueError("Un registro cerrado no puede ser EXCLUIR y CONSERVAR a la vez.")
    original = por_id.loc[identificador]
    if any(grupo[c].ne(original[c]).any() for c in columnas):
        raise ValueError("Se alteraron los campos originales de la revisión.")
    for campo in ["Evidencia", "Comentario"]:
        if grupo[campo].str.strip().eq("").any():
            raise ValueError(f"Una revisión cerrada requiere {campo}.")
    if estados == {"EXCLUIR"}:
        if len(grupo) != 1 or any(grupo[c].str.strip().ne("").any() for c in ["Posicion_autor", "Autor_UNAM", "Afiliacion1_UNAM", "Afiliacion2_UNAM"]):
            raise ValueError("EXCLUIR se registra una vez, sin autores ni afiliaciones de salida.")
        cierres_revision.append({"ID_revision": identificador, "Decision": "EXCLUIR", "Filas": 0})
        continue
    nombres = [nombre_separado(a, original["Fuente_origen"].strip()) for a in separar_punto_coma(original["Autor_norm"])]
    usadas = set()
    for _, decision in grupo.iterrows():
        posicion = decision["Posicion_autor"].strip()
        if not re.fullmatch(r"[1-9]\d*", posicion) or not 1 <= int(posicion) <= len(nombres):
            raise ValueError("Posicion_autor debe identificar al autor en la lista original (desde 1).")
        if posicion in usadas:
            raise ValueError("Hay dos decisiones de conservación para la misma aparición de autor.")
        usadas.add(posicion)
        autor = decision["Autor_UNAM"]
        if autor != nombres[int(posicion)-1]:
            raise ValueError("Autor_UNAM debe ser el nombre separado original; normalizar nombres pertenece al 04.")
        if not re.fullmatch(r"https?://[^\s]+", decision["URL_evidencia"].strip()):
            raise ValueError("Una conservación manual exige URL de evidencia bibliográfica.")
        af1, af2 = decision["Afiliacion1_UNAM"], decision["Afiliacion2_UNAM"]
        if not af1.strip() or (af2 and decodificar_html(af1).casefold() == decodificar_html(af2).casefold()):
            raise ValueError("Indica una o dos afiliaciones UNAM distintas, una por columna.")
        for afiliacion in [af1, af2]:
            if not afiliacion:
                continue
            estado = clasificar_afiliacion(afiliacion)[0]
            if estado != "UNAM":
                raise ValueError("Afiliación manual no reconocida como UNAM: aporta denominación inequívoca o clasificación verificada.")
        fila = {c: original[c] for c in columnas}
        fila.update(Autor_norm=autor, Afiliacion1=af1, Afiliacion2=af2, _row_id=original["_row_id"])
        manual_aprobado.append(fila)
    cierres_revision.append({"ID_revision": identificador, "Decision": "CONSERVAR", "Filas": len(grupo)})

manual = pd.DataFrame(manual_aprobado, columns=columnas + ["_row_id"])
validar_salida(manual)
autores_unam_parcial = pd.concat([autores_unam_automatico, manual], ignore_index=True)
validar_salida(autores_unam_parcial)

# No presentar una base incompleta como final. No deduplicar ni renumerar.
autores_unam_separados = None
if not ids_pendientes:
    autores_unam_separados = autores_unam_parcial[columnas].copy().reset_index(drop=True)
    preparar_csv(autores_unam_separados)
    print("Revisión cerrada. Base completa disponible EN MEMORIA:", autores_unam_separados.shape)
else:
    print("Revisión pendiente de", len(ids_pendientes), "registros. Base final no disponible.")

## 16. RESUMEN FINAL

In [ ]:
# Únicamente estos TRES CSV. No se crean más archivos ni subcarpetas.
os.makedirs(carpeta_salida, exist_ok=True)
guardar_csv(clasificacion_final, archivo_clasificacion)
guardar_csv(casos_revision, revision_manual)
# Se guarda aunque haya casos pendientes. No incluye filas aprobadas manualmente.
automatico_guardado = guardar_csv(salida_automatica_df, salida_automatica)

hash_despues = sha256(archivo)
if hash_despues != hash_antes:
    raise ValueError("El archivo original cambió.")
print("\n=== RESUMEN FINAL ===")
print("Filas de entrada:", len(df))
print("Registros originales para revisión:", len(filas_revision))
print("  Casos grandes (>10):", int(df["_caso_grande"].sum()))
print("  Separación no resuelta:", len(filas_pendientes))
print("  Más de dos afiliaciones UNAM:", len(filas_mas_dos))
print("Filas UNAM automáticas:", len(salida_automatica_df))
print("Filas UNAM manuales aprobadas (en memoria):", len(manual))
print("Relaciones externas excluidas automáticamente:", int(relaciones_automaticas["Estado_UNAM"].eq("EXTERNO").sum()))
print("Afiliaciones externas retiradas de autores conservados:", sum(
    r["Clasificaciones"].count("EXTERNO") for _, r in relaciones_unam.iterrows()
))
print("Revisiones pendientes:", len(ids_pendientes))
print("Columnas originales de la salida automática:", len(salida_automatica_df.columns))
print("Archivo original intacto:", hash_antes == hash_despues)
print("Clasificación:", archivo_clasificacion)
print("Casos manuales:", revision_manual)
if automatico_guardado:
    print("Salida automática guardada/verificada:", salida_automatica)
else:
    print("Salida automática NO actualizada: activa actualizar_archivos para regenerarla.")
print("No se generaron archivos adicionales.")